# Local-stim response analysis (exp_426)

Loads the exported zarr + `exp_data.parquet` from a random-patch stim run and:

1. Pulls the mean intensity inside each cell's 14 px stim patch (both imaging channels), per timepoint, and merges back onto `df_exp`.
2. Provides cell-level QC filters (mean expression per channel, area).
3. Picks a deterministic random *control* patch from the same cell — same selection algorithm but excluding the stim patch — and measures intensity there too, so each cell carries its own unstimulated baseline.
4. Plots per-channel mean response over time for stim vs. control patches.
5. Extracts 128 px windows around each stim patch and computes texture / shape features so we can correlate local response with cell structure.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import sobel

# ---- Edit for the run ----
BASE_PATH       = r"Z:\lhinder\data\rtm_mm_data\exp_427"
EXPERIMENT_NAME = "random_stim_per_cell_14px_patches"
PATCH_SIZE      = 14   # must match the stimulator (RandomStimPerCell14pxPatches.PATCH_SIZE)
DOT_DIAMETER    = 10   # ditto, for the actually-illuminated dot inside each patch
TEXTURE_PATCH   = 52  # square window around the stim patch for texture feats

# Imaging channel names *in the order they appear on the C axis of the raw zarr*.
# Order = imaging channels first (as passed to the experiment), then stim readouts.
IMAGING_CHANNELS = ["mScarlet3", "miRFP"]

path = Path(BASE_PATH) / EXPERIMENT_NAME
print("experiment dir:", path)
print("exists:", path.is_dir())

In [ ]:
df_exp = pd.read_parquet(path / "exp_data.parquet")
print(df_exp.shape)
print(df_exp.columns.tolist())
df_exp.head()

In [ ]:
# OmeZarrWriter direct-mode layout: <root>/0 has shape (T, P, C, Y, X).
# Labels live at <root>/labels/<name>/0 with shape (T, P, Y, X).
store = zarr.open(str(path / "acquisition.ome.zarr"), mode="r")
raw   = store["0"]
print("raw shape:", raw.shape, "dtype:", raw.dtype)

labels = None
for name in ("particles", "labels"):  # prefer tracked labels if present
    try:
        labels = store[f"labels/{name}/0"]
        print(f"labels: {name}/0 shape={labels.shape}")
        break
    except KeyError:
        continue
if labels is None:
    raise RuntimeError("No labels array found under labels/particles or labels/labels.")

T, P, C, H, W = raw.shape
ch_idx = {name: i for i, name in enumerate(IMAGING_CHANNELS)}
print("T, P, C, H, W =", T, P, C, H, W)
print("imaging channel index map:", ch_idx)

### Labels frame cache

`get_labels_frame(t, p)` lazy-loads + caches the (Y, X) label array for a
given (timepoint, FOV). Used by the cell-overlay step (centroid lookup) and
the control-patch step (re-running the stimulator's eligibility check).


In [ ]:
_label_cache: dict[tuple[int, int], np.ndarray] = {}

def get_labels_frame(t: int, p: int) -> np.ndarray:
    key = (t, p)
    if key not in _label_cache:
        _label_cache[key] = np.asarray(labels[t, p])
    return _label_cache[key]


## 1. Mean intensity inside each cell's stim patch

`df_exp` has one row per (fov, fov_timestep, label) and the stim-frame rows carry `patch_y_min`/`patch_x_min`/`patch_y_max`/`patch_x_max` (the 14 px tile) and `patch_dot_y`/`patch_dot_x` (the 10 px illuminated dot center). We measure:

- **Patch mean**: average over the full 14×14 tile.
- **Dot mean**: average over the 10 px circle that is actually hit by the DMD.

Dot mean is the better readout of the stim response (whole-patch mean dilutes with dark non-illuminated pixels), but both are recorded so you can choose.

In [ ]:
def _dot_mask(ps: int, diameter: float) -> np.ndarray:
    center = (ps - 1) / 2
    r = diameter / 2
    yy, xx = np.ogrid[:ps, :ps]
    return ((yy - center) ** 2 + (xx - center) ** 2) <= r ** 2

DOT_MASK = _dot_mask(PATCH_SIZE, DOT_DIAMETER)


def patch_means(t: int, p: int, y0: int, x0: int) -> dict:
    """Read a 14x14 tile from each imaging channel at (t, p) and return
    {channel_name: (patch_mean, dot_mean)}."""
    y1, x1 = y0 + PATCH_SIZE, x0 + PATCH_SIZE
    if y0 < 0 or x0 < 0 or y1 > H or x1 > W:
        return {ch: (np.nan, np.nan) for ch in IMAGING_CHANNELS}
    out = {}
    for ch, ci in ch_idx.items():
        tile = np.asarray(raw[t, p, ci, y0:y1, x0:x1])
        out[ch] = (float(tile.mean()), float(tile[DOT_MASK].mean()))
    return out

# Apply to stim-frame rows only (the only ones with patch coords).
stim_rows = df_exp.dropna(subset=["patch_y_min", "patch_x_min"]).copy()
print(f"{len(stim_rows)} stim-frame rows")

for ch in IMAGING_CHANNELS:
    df_exp[f"stim_patch_mean_{ch}"] = np.nan
    df_exp[f"stim_dot_mean_{ch}"]   = np.nan

for idx, row in stim_rows.iterrows():
    t = int(row["fov_timestep"])
    p = int(row["fov"])
    y0 = int(row["patch_y_min"]); x0 = int(row["patch_x_min"])
    means = patch_means(t, p, y0, x0)
    for ch, (pm, dm) in means.items():
        df_exp.loc[idx, f"stim_patch_mean_{ch}"] = pm
        df_exp.loc[idx, f"stim_dot_mean_{ch}"]   = dm

df_exp[[c for c in df_exp.columns if c.startswith("stim_")]].describe()

### Sanity check: stim patch overlaid on the raw image

Picks a few random stim-frame rows and shows the imaging channel with the 14 px
stim patch and the 10 px dot drawn on top. If the patch lands on a cell, the
coordinate convention is correct.


In [ ]:
import matplotlib.patches as mpatches

n_show = 6
sample = stim_rows.sample(min(n_show, len(stim_rows)), random_state=0)
fig, axs = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.4))
if len(sample) == 1:
    axs = [axs]

for ax, (_, row) in zip(axs, sample.iterrows()):
    t  = int(row["fov_timestep"])
    p  = int(row["fov"])
    y0 = int(row["patch_y_min"]); x0 = int(row["patch_x_min"])
    dy = float(row["patch_dot_y"]); dx = float(row["patch_dot_x"])
    img = np.asarray(raw[t, p, ch_idx[IMAGING_CHANNELS[0]]])
    ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    ax.add_patch(mpatches.Rectangle((x0, y0), PATCH_SIZE, PATCH_SIZE,
                                    edgecolor="red", facecolor="none", linewidth=1.0))
    ax.add_patch(mpatches.Circle((dx, dy), DOT_DIAMETER / 2,
                                 edgecolor="yellow", facecolor="none", linewidth=1.0))
    ax.set_title(f"fov={p} t={t} label={int(row['label'])}", fontsize=8)
    ax.axis("off")
fig.suptitle("stim patch (red) + dot (yellow) on imaging channel 0")
fig.tight_layout()


## 2. Cell-level QC filters

Per-cell quality summary based on whatever the feature extractor stored on `df_exp` plus per-cell area derived from the label array. Knobs:

- `min_mean_intensity_<channel>` — drop cells whose mean intensity (averaged across all frames they appear in) is below this. Filters out non-expressers.
- `min_area_px` / `max_area_px` — drop cells outside this size range.

We use the existing `mean_intensity_<channel>` column from feature extraction if present; otherwise we derive a per-frame per-cell mean from the label array.

In [ ]:
# Per-cell stats: index by (fov, particle) if tracked, else (fov, label).
# 'particle' is the trackpy particle id and stays stable across frames; 'label'
# is the per-frame segmentation id and may collide across frames. Prefer particle.
cell_key = "particle" if "particle" in df_exp.columns else "label"
print(f"grouping cells by ({cell_key}, fov)")


def _find_intensity_col(df, channel_name: str, channel_idx: int) -> str | None:
    """Find a per-frame intensity column for a given imaging channel.

    FE_ErkKtr names columns by channel *index* (``mean_intensity_C0_nuc``,
    ``mean_intensity_C1_ring``, ...); other extractors may use the channel
    *name* (``mean_intensity_mScarlet3``). Try the most informative options
    in order, then fall back to any ``mean_intensity*`` column that
    references either the index or the name.
    """
    explicit = [
        f"mean_intensity_C{channel_idx}_nuc",
        f"mean_intensity_C{channel_idx}_ring",
        f"mean_intensity_{channel_name}",
        f"median_intensity_C{channel_idx}_nuc",
    ]
    for c in explicit:
        if c in df.columns:
            return c
    name_l = channel_name.lower()
    idx_marker = f"_c{channel_idx}_"
    for c in df.columns:
        cl = c.lower()
        if cl.startswith(("mean_intensity", "median_intensity", "intensity_mean")) and (
            name_l in cl or idx_marker in cl
        ):
            return c
    return None


intensity_cols = {
    ch: _find_intensity_col(df_exp, ch, i)
    for i, ch in enumerate(IMAGING_CHANNELS)
}
area_col = next(
    (c for c in ("area", "area_nuc", "cell_area") if c in df_exp.columns),
    None,
)
print("per-frame intensity columns found:", intensity_cols)
print("area column found:", area_col)

agg_dict: dict[str, str] = {}
rename_map: dict[str, str] = {}
for ch, col in intensity_cols.items():
    if col is not None:
        agg_dict[col] = "mean"
        rename_map[col] = f"cellmean_{ch}"
if area_col is not None:
    agg_dict[area_col] = "mean"
    rename_map[area_col] = "cellmean_area"

if agg_dict:
    cell_stats = (
        df_exp.groupby(["fov", cell_key])
        .agg(agg_dict)
        .rename(columns=rename_map)
        .reset_index()
    )
else:
    print("[warning] no intensity/area columns found — skipping QC, keeping all cells.")
    cell_stats = df_exp[["fov", cell_key]].drop_duplicates().reset_index(drop=True)

# How many distinct frames each cell appears in (uses df_exp directly because
# cell_stats has been reduced to one row per cell already).
n_frames_total = int(df_exp["fov_timestep"].nunique())
frames_per_cell = (
    df_exp.groupby(["fov", cell_key])["fov_timestep"].nunique()
          .reset_index(name="n_frames_tracked")
)
cell_stats = cell_stats.merge(frames_per_cell, on=["fov", cell_key], how="left")

# Whether each cell got at least one stim patch (i.e. cellpose found it on a
# stim frame and the stimulator picked a fully-inside patch for it).
cells_with_stim = (
    df_exp.dropna(subset=["patch_y_min"])[["fov", cell_key]]
          .drop_duplicates()
)
cells_with_stim["has_stim_area"] = True
cell_stats = cell_stats.merge(cells_with_stim, on=["fov", cell_key], how="left")
cell_stats["has_stim_area"] = cell_stats["has_stim_area"].fillna(False)

# ---- Edit the thresholds for your run ----
thresholds = {
    "min_mean_intensity": {"mScarlet3": 200.0, "miRFP": 800.0},  # drop non-expressers
    "min_area_px": 2000,
    "max_area_px": 12000,
    "min_frame_fraction": 0.95,  # cell must appear in >= this fraction of frames
    "require_stim_area": True,   # cell must have at least one stim patch
}

keep_mask = pd.Series(True, index=cell_stats.index)
for ch, lo in thresholds["min_mean_intensity"].items():
    col = f"cellmean_{ch}"
    if col in cell_stats.columns:
        keep_mask &= cell_stats[col] >= lo
if "cellmean_area" in cell_stats.columns:
    keep_mask &= cell_stats["cellmean_area"] >= thresholds["min_area_px"]
    keep_mask &= cell_stats["cellmean_area"] <= thresholds["max_area_px"]
if thresholds.get("min_frame_fraction"):
    min_frames_req = thresholds["min_frame_fraction"] * n_frames_total
    keep_mask &= cell_stats["n_frames_tracked"] >= min_frames_req
if thresholds.get("require_stim_area"):
    keep_mask &= cell_stats["has_stim_area"]

kept = cell_stats.loc[keep_mask, ["fov", cell_key]].drop_duplicates()
print(
    f"kept {len(kept)} / {len(cell_stats)} cells after QC "
    f"(n_frames_total={n_frames_total}, "
    f"min_frames={int(thresholds['min_frame_fraction'] * n_frames_total) if thresholds.get('min_frame_fraction') else 0})"
)
df_keep = df_exp.merge(kept, on=["fov", cell_key], how="inner")


### 10 random filtered cells at the first stim frame

Both imaging channels for the same 200 x 200 px crop centered on each cell.
Red = the 14 px stim patch (where the DMD actually fires); cyan dashed = the
128 px texture window used by the texture cell below. Title shows the FOV and
each channel's per-cell mean intensity (computed in the QC cell above), so the
filter thresholds can be sanity-checked visually.


In [ ]:
import matplotlib.patches as mpatches

CROP = 200
HALF = CROP // 2

# Derive stim_frames locally so this cell doesn't depend on the plot cell.
stim_frames_local = sorted(stim_rows["fov_timestep"].unique().astype(int).tolist())
first_stim = stim_frames_local[0] if stim_frames_local else 0

sample = kept.sample(min(10, len(kept)), random_state=0).reset_index(drop=True)

stats_idx = (
    cell_stats.set_index(["fov", cell_key])
    if any(c.startswith("cellmean_") for c in cell_stats.columns)
    else None
)

n_show = len(sample)
fig, axs = plt.subplots(
    len(IMAGING_CHANNELS), n_show,
    figsize=(2.6 * n_show, 2.6 * len(IMAGING_CHANNELS)),
    squeeze=False,
)

for col_i, (_, srow) in enumerate(sample.iterrows()):
    p   = int(srow["fov"])
    cid = int(srow[cell_key])

    stim_match = df_exp[
        (df_exp["fov"] == p)
        & (df_exp[cell_key] == cid)
        & (df_exp["fov_timestep"] == first_stim)
        & df_exp["patch_y_min"].notna()
    ]
    if stim_match.empty:
        stim_match = df_exp[
            (df_exp["fov"] == p)
            & (df_exp[cell_key] == cid)
            & df_exp["patch_y_min"].notna()
        ]
    if stim_match.empty:
        for ax_row in axs:
            ax_row[col_i].axis("off")
        continue
    row = stim_match.iloc[0]
    t_stim = int(row["fov_timestep"])

    # Labels image is `labels/particles/0` (per the open-zarr cell preference),
    # so its values are particle ids. Look up by cid (particle id from
    # cell_key), NOT by row["label"] (per-frame segmentation id).
    lbl_frame = get_labels_frame(t_stim, p)
    yy, xx = np.where(lbl_frame == cid)
    if len(yy) == 0:
        for ax_row in axs:
            ax_row[col_i].axis("off")
        continue
    cy = int(yy.mean()); cx = int(xx.mean())

    y0 = max(0, cy - HALF); y1 = min(H, y0 + CROP); y0 = max(0, y1 - CROP)
    x0 = max(0, cx - HALF); x1 = min(W, x0 + CROP); x0 = max(0, x1 - CROP)

    py0 = float(row["patch_y_min"]) - y0
    px0 = float(row["patch_x_min"]) - x0
    dy  = float(row["patch_dot_y"]) - y0
    dx  = float(row["patch_dot_x"]) - x0
    tex_y = dy - TEXTURE_PATCH / 2
    tex_x = dx - TEXTURE_PATCH / 2

    ann_lines = [f"FOV {p}  cell {cid}"]
    if stats_idx is not None and (p, cid) in stats_idx.index:
        s = stats_idx.loc[(p, cid)]
        for ch in IMAGING_CHANNELS:
            v = s.get(f"cellmean_{ch}")
            if v is not None and not pd.isna(v):
                ann_lines.append(f"{ch}: {float(v):.0f}")

    for row_i, ch in enumerate(IMAGING_CHANNELS):
        ax = axs[row_i, col_i]
        img = np.asarray(raw[t_stim, p, ch_idx[ch], y0:y1, x0:x1])
        lo, hi = np.percentile(img, [1, 99.99])
        ax.imshow(img, cmap="gray_r", vmin=lo, vmax=hi)
        ax.add_patch(mpatches.Rectangle((px0, py0), PATCH_SIZE, PATCH_SIZE,
                                        edgecolor="red", facecolor="none", linewidth=1.0))
        ax.add_patch(mpatches.Circle((dx, dy), DOT_DIAMETER / 2,
                                     edgecolor="yellow", facecolor="none", linewidth=0.8))
        ax.add_patch(mpatches.Rectangle((tex_x, tex_y), TEXTURE_PATCH, TEXTURE_PATCH,
                                        edgecolor="cyan", facecolor="none",
                                        linewidth=0.7, linestyle="--"))
        ax.set_xlim(0, x1 - x0); ax.set_ylim(y1 - y0, 0)
        ax.set_xticks([]); ax.set_yticks([])
        if row_i == 0:
            ax.set_title("\n".join(ann_lines), fontsize=8)
        if col_i == 0:
            ax.set_ylabel(ch, fontsize=9)

fig.suptitle(
    f"10 random filtered cells at t={first_stim}  "
    f"(red = stim patch, cyan dashed = 128 px texture window)"
)
fig.tight_layout()


## 3. Random control patch from the same cell

For each stim-frame row we re-run the stimulator's patch-selection algorithm on the labels image, exclude the stim patch from the candidate set, and pick another random patch *fully inside the same cell* with a deterministic seed (so the control is reproducible).

Then we read the same intensity stats inside that control patch, giving each cell its own paired stim/control timecourse.

In [ ]:
def candidate_patches_for_label(labels_frame: np.ndarray, cell_id: int) -> np.ndarray:
    """Replicate the stimulator's per-cell patch eligibility check: tile the
    image into PATCH_SIZE blocks and keep blocks whose every pixel == cell_id.
    Returns array of (i, j) grid indices.

    NOTE: ``cell_id`` must match the value space of ``labels_frame`` — when
    the cached labels array is ``labels/particles/0`` (tracked particle ids,
    the default in the open-zarr cell), pass the particle id; the
    per-frame segmentation label from ``row["label"]`` will not match.
    """
    ps = PATCH_SIZE
    h, w = labels_frame.shape
    n_h, n_w = h // ps, w // ps
    if n_h == 0 or n_w == 0:
        return np.empty((0, 2), dtype=int)
    cropped = labels_frame[: n_h * ps, : n_w * ps]
    blocks = cropped.reshape(n_h, ps, n_w, ps)
    block_min = blocks.min(axis=(1, 3))
    block_max = blocks.max(axis=(1, 3))
    uniform = (block_min == block_max) & (block_min == cell_id)
    ii, jj = np.nonzero(uniform)
    return np.stack([ii, jj], axis=1)


def control_patch_for_row(row, exclude_seed: int = 7) -> tuple[int, int] | None:
    """Return (patch_y_min, patch_x_min) of a random in-cell patch != stim patch,
    using a seed derived from (fov, cell_id) for reproducibility."""
    t  = int(row["fov_timestep"])
    p  = int(row["fov"])
    cid = int(row[cell_key])  # particle id — matches labels/particles/0 values
    cands = candidate_patches_for_label(get_labels_frame(t, p), cid)
    if len(cands) < 2:
        return None
    si = int(row["patch_i"]); sj = int(row["patch_j"])
    mask = ~((cands[:, 0] == si) & (cands[:, 1] == sj))
    cands = cands[mask]
    if len(cands) == 0:
        return None
    seed = (p * 1_000_003 + cid * 100_003 + exclude_seed) & 0xFFFFFFFF
    rng = np.random.default_rng(seed)
    pi, pj = cands[rng.integers(len(cands))]
    return int(pi) * PATCH_SIZE, int(pj) * PATCH_SIZE


for ch in IMAGING_CHANNELS:
    df_exp[f"ctrl_patch_mean_{ch}"] = np.nan
    df_exp[f"ctrl_dot_mean_{ch}"]   = np.nan
df_exp["ctrl_patch_y_min"] = np.nan
df_exp["ctrl_patch_x_min"] = np.nan

for idx, row in stim_rows.iterrows():
    ctrl = control_patch_for_row(row)
    if ctrl is None:
        continue
    cy, cx = ctrl
    df_exp.loc[idx, "ctrl_patch_y_min"] = cy
    df_exp.loc[idx, "ctrl_patch_x_min"] = cx
    means = patch_means(int(row["fov_timestep"]), int(row["fov"]), cy, cx)
    for ch, (pm, dm) in means.items():
        df_exp.loc[idx, f"ctrl_patch_mean_{ch}"] = pm
        df_exp.loc[idx, f"ctrl_dot_mean_{ch}"]   = dm

n_with_ctrl = df_exp["ctrl_patch_y_min"].notna().sum()
print(f"{n_with_ctrl} / {len(stim_rows)} stim rows got a control patch")
df_exp[[c for c in df_exp.columns if c.startswith("ctrl_")]].describe()


## 3b. Extend the readout to baseline + recovery frames

Patch selection is fixed per cell (one selection per FOV, applied across all
stim frames), so the same `(patch_y_min, patch_x_min)` and the control patch
location are valid at every timepoint. We propagate them from the stim rows
to every row of the same cell and re-extract intensities so the timecourse
covers baseline + stim + recovery, not just the stim block.

We then normalize each cell's per-frame stim-dot intensity by its own
**baseline median** (median across the pre-stim frames). The resulting
`stim_dot_fold_<channel>` is the fold-change from the cell's baseline — the
shape of the response is comparable across cells with different absolute
expression levels.


In [ ]:
# 1) Propagate patch coords from the stim rows to every row of the same cell.
patch_propagate_cols = [
    "patch_y_min", "patch_x_min", "patch_y_max", "patch_x_max",
    "patch_dot_y", "patch_dot_x", "patch_i", "patch_j",
    "ctrl_patch_y_min", "ctrl_patch_x_min",
]
patch_propagate_cols = [c for c in patch_propagate_cols if c in df_exp.columns]

src_coords = (
    df_exp.dropna(subset=["patch_y_min"])
          .groupby(["fov", cell_key])[patch_propagate_cols]
          .first()
          .reset_index()
)
df_exp = df_exp.drop(columns=patch_propagate_cols).merge(
    src_coords, on=["fov", cell_key], how="left"
)

# 2) Re-read intensities for every (cell, frame) row that now has coords.
# Sort by (fov, t) and keep one (C, H, W) image stack in RAM per (t, p) so each
# zarr frame is fetched once instead of once per cell. ~70x fewer zarr reads.
all_rows = df_exp.dropna(subset=["patch_y_min"]).copy()
print(f"re-extracting intensity for {len(all_rows)} (cell, frame) rows")

for ch in IMAGING_CHANNELS:
    df_exp[f"stim_patch_mean_{ch}"] = np.nan
    df_exp[f"stim_dot_mean_{ch}"]   = np.nan
    df_exp[f"ctrl_patch_mean_{ch}"] = np.nan
    df_exp[f"ctrl_dot_mean_{ch}"]   = np.nan

n_img_ch = len(IMAGING_CHANNELS)
all_rows_sorted = all_rows.sort_values(["fov", "fov_timestep"])
cur_key: tuple[int, int] | None = None
cur_imgs: np.ndarray | None = None
processed = 0
report_every = 2000

for orig_idx, row in all_rows_sorted.iterrows():
    t = int(row["fov_timestep"]); p = int(row["fov"])
    key = (t, p)
    if key != cur_key:
        cur_imgs = np.asarray(raw[t, p, :n_img_ch])  # (C, H, W) uint16
        cur_key = key

    sy0 = int(row["patch_y_min"]); sx0 = int(row["patch_x_min"])
    if 0 <= sy0 and sy0 + PATCH_SIZE <= H and 0 <= sx0 and sx0 + PATCH_SIZE <= W:
        for ci, ch in enumerate(IMAGING_CHANNELS):
            tile = cur_imgs[ci, sy0:sy0 + PATCH_SIZE, sx0:sx0 + PATCH_SIZE]
            df_exp.at[orig_idx, f"stim_patch_mean_{ch}"] = float(tile.mean())
            df_exp.at[orig_idx, f"stim_dot_mean_{ch}"]   = float(tile[DOT_MASK].mean())

    cy0v = row.get("ctrl_patch_y_min"); cx0v = row.get("ctrl_patch_x_min")
    if pd.notna(cy0v) and pd.notna(cx0v):
        cy0i = int(cy0v); cx0i = int(cx0v)
        if 0 <= cy0i and cy0i + PATCH_SIZE <= H and 0 <= cx0i and cx0i + PATCH_SIZE <= W:
            for ci, ch in enumerate(IMAGING_CHANNELS):
                tile = cur_imgs[ci, cy0i:cy0i + PATCH_SIZE, cx0i:cx0i + PATCH_SIZE]
                df_exp.at[orig_idx, f"ctrl_patch_mean_{ch}"] = float(tile.mean())
                df_exp.at[orig_idx, f"ctrl_dot_mean_{ch}"]   = float(tile[DOT_MASK].mean())

    processed += 1
    if processed % report_every == 0:
        print(f"  {processed}/{len(all_rows)} rows...")

print(f"  done ({processed} rows)")

# 3) Per-cell baseline median, used to compute fold-change.
stim_frames_local = sorted(
    df_exp.dropna(subset=["patch_y_min"])["fov_timestep"].unique().astype(int).tolist()
)
baseline_frames_list = list(range(0, min(stim_frames_local) if stim_frames_local else 0))
print("baseline frames used for normalization:", baseline_frames_list)

dot_cols = [
    f"{prefix}_dot_mean_{ch}"
    for prefix in ("stim", "ctrl")
    for ch in IMAGING_CHANNELS
]
baseline_med = (
    df_exp[df_exp["fov_timestep"].isin(baseline_frames_list)]
    .groupby(["fov", cell_key])[dot_cols]
    .median()
    .add_suffix("_baseline_med")
    .reset_index()
)
df_exp = df_exp.drop(
    columns=[c for c in df_exp.columns if c.endswith("_baseline_med")],
    errors="ignore",
).merge(baseline_med, on=["fov", cell_key], how="left")

for prefix in ("stim", "ctrl"):
    for ch in IMAGING_CHANNELS:
        col  = f"{prefix}_dot_mean_{ch}"
        bcol = f"{col}_baseline_med"
        df_exp[f"{prefix}_dot_fold_{ch}"] = df_exp[col] / df_exp[bcol]

print("added columns:")
for c in df_exp.columns:
    if c.endswith("_baseline_med") or "_dot_fold_" in c:
        print(" ", c)


## 4. Stim vs. control intensity over time

Per-channel mean ± SEM across all kept (filtered) cells, plotted against `fov_timestep`. The shaded stim window highlights when the DMD was active.

In [ ]:
df_plot = df_exp.merge(kept, on=["fov", cell_key], how="inner")
stim_frames = sorted(
    df_plot.dropna(subset=["patch_y_min"])["fov_timestep"].unique().astype(int)
)
print("detected stim frames:", stim_frames)

# Raw dot-mean (left col) + per-cell fold-change (right col), one row per channel.
fig, axs = plt.subplots(
    len(IMAGING_CHANNELS), 2,
    figsize=(11, 3.0 * len(IMAGING_CHANNELS)),
    sharex=True, squeeze=False,
)

for r, ch in enumerate(IMAGING_CHANNELS):
    grp = df_plot.groupby("fov_timestep")
    for c, (suffix, ylabel) in enumerate(
        [("mean", f"{ch} dot mean (raw)"),
         ("fold", f"{ch} dot fold (vs baseline)")]
    ):
        scol = f"stim_dot_{suffix}_{ch}"
        ccol = f"ctrl_dot_{suffix}_{ch}"
        ax = axs[r, c]
        if scol not in df_plot.columns or ccol not in df_plot.columns:
            ax.set_title(f"{ch}: missing {scol}/{ccol}")
            continue
        sm = grp[scol].mean(); ss = grp[scol].sem()
        cm = grp[ccol].mean(); cs = grp[ccol].sem()
        ax.plot(sm.index, sm.values, color="C3", label="stim")
        ax.fill_between(sm.index, sm - ss, sm + ss, color="C3", alpha=0.2)
        ax.plot(cm.index, cm.values, color="C0", label="control")
        ax.fill_between(cm.index, cm - cs, cm + cs, color="C0", alpha=0.2)
        if stim_frames:
            ax.axvspan(20, 30,
                       color="yellow", alpha=0.15, label="stim window")
        if suffix == "fold":
            ax.axhline(1.0, color="gray", lw=0.6, ls=":")
        ax.set_ylabel(ylabel)
        if r == 0 and c == 0:
            ax.legend(loc="upper right", fontsize=8)
        ax.set_xlim(0,80)


axs[-1, 0].set_xlabel("fov_timestep")
axs[-1, 1].set_xlabel("fov_timestep")
axs[-1,0].set_ylim(800,1400)
axs[-2,0].set_ylim(10000, 45400)

fig.tight_layout()


## 5. Local cell-structure features around each stim patch

For every stim-frame row, extract a `TEXTURE_PATCH × TEXTURE_PATCH` window centered on the stim dot from each imaging channel, then compute:

- **Intensity stats**: mean, std, P5–P95 range.
- **Edge strength**: mean Sobel magnitude (proxy for fibrousness vs. smooth interior).
- **GLCM stats** (gray-level co-occurrence, 8-bit quantized): contrast, homogeneity, energy, correlation — a basic texture fingerprint that separates uniform from streaky/fibrous patches.

Compute on the channel that carries the relevant structure — usually `miRFP` (the PIP/structural label). Adjust `TEXTURE_CHANNEL` if you want texture from `mScarlet3` instead.

In [ ]:
TEXTURE_CHANNEL = "miRFP"   # which imaging channel to compute texture on
GLCM_LEVELS = 32            # quantization for graycomatrix (8-32 is typical)

def extract_window(t: int, p: int, ci: int, cy: int, cx: int, size: int) -> np.ndarray:
    """Return a (size, size) window centered at (cy, cx) with zero-padding at edges."""
    half = size // 2
    y0, y1 = cy - half, cy + size - half
    x0, x1 = cx - half, cx + size - half
    yy0 = max(0, y0); xx0 = max(0, x0)
    yy1 = min(H, y1); xx1 = min(W, x1)
    out = np.zeros((size, size), dtype=raw.dtype)
    if yy0 >= yy1 or xx0 >= xx1:
        return out
    src = np.asarray(raw[t, p, ci, yy0:yy1, xx0:xx1])
    dy0 = yy0 - y0; dx0 = xx0 - x0
    out[dy0:dy0 + src.shape[0], dx0:dx0 + src.shape[1]] = src
    return out


def texture_features(window: np.ndarray) -> dict:
    feats = {}
    if window.size == 0 or window.max() == 0:
        return {k: np.nan for k in (
            "mean", "std", "p5_p95_range", "sobel_mean",
            "glcm_contrast", "glcm_homogeneity", "glcm_energy", "glcm_correlation",
        )}
    w = window.astype(np.float32)
    feats["mean"] = float(w.mean())
    feats["std"]  = float(w.std())
    p5, p95 = np.percentile(w, [5, 95])
    feats["p5_p95_range"] = float(p95 - p5)
    feats["sobel_mean"]   = float(sobel(w / max(1.0, w.max())).mean())

    # GLCM on min-max-quantized window. Average over 4 directions.
    lo, hi = w.min(), w.max()
    if hi > lo:
        q = ((w - lo) / (hi - lo) * (GLCM_LEVELS - 1)).astype(np.uint8)
    else:
        q = np.zeros_like(w, dtype=np.uint8)
    glcm = graycomatrix(
        q, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=GLCM_LEVELS, symmetric=True, normed=True,
    )
    for prop in ("contrast", "homogeneity", "energy", "correlation"):
        feats[f"glcm_{prop}"] = float(graycoprops(glcm, prop).mean())
    return feats


tex_ci = ch_idx[TEXTURE_CHANNEL]
feat_keys = (
    "mean", "std", "p5_p95_range", "sobel_mean",
    "glcm_contrast", "glcm_homogeneity", "glcm_energy", "glcm_correlation",
)
for k in feat_keys:
    df_exp[f"tex_{TEXTURE_CHANNEL}_{k}"] = np.nan

for idx, row in stim_rows.iterrows():
    cy = int(round(row["patch_dot_y"]))
    cx = int(round(row["patch_dot_x"]))
    win = extract_window(int(row["fov_timestep"]), int(row["fov"]),
                         tex_ci, cy, cx, TEXTURE_PATCH)
    fts = texture_features(win)
    for k, v in fts.items():
        df_exp.loc[idx, f"tex_{TEXTURE_CHANNEL}_{k}"] = v

df_exp[[c for c in df_exp.columns if c.startswith(f'tex_{TEXTURE_CHANNEL}_')]].describe()

### Quick scatter: response amplitude vs. local texture

Define each cell's *response amplitude* as the difference between (mean stim-window dot intensity) and (mean baseline dot intensity), then plot vs. each texture feature. A non-flat trend says local structure modulates response.

In [ ]:
RESPONSE_CHANNEL = "miRFP"   # which channel's intensity to treat as the response readout
BASELINE_FRAMES  = range(0, min(stim_frames) if stim_frames else 0)
STIM_WINDOW      = stim_frames

# IMPORTANT: re-merge from the *current* df_exp, not from the df_plot snapshot
# made in the plot cell — that snapshot was taken before the texture columns
# were added in the previous cell, so df_plot does not have any tex_* columns
# and the groupby[tex_cols] selector below would KeyError on them.
df_resp = df_exp.merge(kept, on=["fov", cell_key], how="inner")
df_resp["is_baseline"] = df_resp["fov_timestep"].isin(list(BASELINE_FRAMES))
df_resp["is_stim"]     = df_resp["fov_timestep"].isin(list(STIM_WINDOW))

stim_col = f"stim_dot_mean_{RESPONSE_CHANNEL}"
ctrl_col = f"ctrl_dot_mean_{RESPONSE_CHANNEL}"
amp = (
    df_resp[df_resp["is_stim"]]
    .groupby(["fov", cell_key])
    .agg(stim=(stim_col, "mean"), ctrl=(ctrl_col, "mean"))
    .assign(amplitude=lambda d: d["stim"] - d["ctrl"])
    .reset_index()
)

# Per-cell texture: mean over the cell's stim frames. Pull tex_cols from the
# refreshed df_resp so we never reference columns that don't exist yet.
tex_cols = [c for c in df_resp.columns if c.startswith(f"tex_{TEXTURE_CHANNEL}_")]
if not tex_cols:
    raise RuntimeError(
        f"no tex_{TEXTURE_CHANNEL}_* columns found on df_resp — did the texture "
        "cell run successfully on the current df_exp?"
    )
tex_per_cell = (
    df_resp[df_resp["is_stim"]]
    .groupby(["fov", cell_key])[tex_cols]
    .mean()
    .reset_index()
)

merged = amp.merge(tex_per_cell, on=["fov", cell_key])
print(f"{len(merged)} cells with both response amplitude and texture features")

n = len(tex_cols)
cols = 4
rows = (n + cols - 1) // cols
fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 3 * rows))
for ax, col in zip(axs.flat, tex_cols):
    ax.scatter(merged[col], merged["amplitude"], s=12, alpha=0.5)
    ax.set_xlabel(col.replace(f"tex_{TEXTURE_CHANNEL}_", ""))
    ax.set_ylabel("amplitude")
    if merged[col].notna().sum() > 1 and merged["amplitude"].notna().sum() > 1:
        try:
            r = merged[[col, "amplitude"]].corr().iloc[0, 1]
            ax.set_title(f"r = {r:+.2f}", fontsize=9)
        except Exception:
            pass
for ax in axs.flat[n:]:
    ax.axis("off")
fig.tight_layout()


## 6. Save extended dataframe

Writes the augmented `df_exp` next to the raw `exp_data.parquet` as `exp_data_with_local.parquet` so this analysis can be reloaded without re-extracting patches.

In [ ]:
out_path = path / "exp_data_with_local.parquet"
df_exp.to_parquet(out_path)
print("wrote:", out_path)